In [2]:
%pip install -q FlagEmbedding langgraph google-genai gradio requests pandas ragas datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
# ============================================================
# 1. IMPORTS AND ENVIRONMENT CONFIGURATION
# ============================================================

import os
import re
import json
import time
import math
import getpass
import statistics

from datetime import datetime, timezone
from typing import TypedDict, List, Dict, Any, Optional

import requests
import pandas as pd
import torch
import gradio as gr

from FlagEmbedding import BGEM3FlagModel, FlagReranker
from google import genai
from google.genai.errors import ServerError, ClientError
from langgraph.graph import StateGraph, START, END


# ============================================================
# 2. CHECK PYTHON AND GPU ENVIRONMENT
# ============================================================

print("Python environment ready")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 3. CONFIGURE GDELT RETRIEVAL API
# ============================================================

# Enter the latest Cloudflare Quick Tunnel URL.
# Example: https://example.trycloudflare.com

API_URL = input(
    "Enter current Cloudflare API URL (https://...trycloudflare.com): "
).strip().rstrip("/")

# Securely enter credentials and remove accidental whitespace.
API_KEY = getpass.getpass(
    "Enter GDELT API key (hidden): "
).strip()

GEMINI_API_KEY = getpass.getpass(
    "Enter Gemini API key (hidden): "
).strip()


# ============================================================
# 4. VALIDATE CONFIGURATION
# ============================================================

if not API_URL.startswith("https://"):
    raise ValueError("Invalid API URL. Enter your HTTPS Cloudflare URL.")

if not API_URL.endswith(".trycloudflare.com"):
    raise ValueError("Expected a Cloudflare Quick Tunnel URL.")

if not API_KEY:
    raise ValueError("GDELT API key cannot be empty.")

if not GEMINI_API_KEY:
    raise ValueError("Gemini API key cannot be empty.")


# ============================================================
# 5. INITIALIZE GEMINI
# ============================================================

client = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_MODEL = "gemini-3.5-flash-lite"


# ============================================================
# 6. CHECK GDELT RETRIEVAL API CONNECTION
# ============================================================

print("\nChecking GDELT Retrieval API connection...")

try:
    health = requests.get(
        f"{API_URL}/health",
        headers={
            "X-API-Key": API_KEY
        },
        timeout=30
    )

    health.raise_for_status()

    health_data = health.json()

    print("\nGDELT Retrieval API connected successfully!")
    print("API health:", health_data)

except requests.exceptions.HTTPError as e:
    print("API returned an HTTP error:", e)
    print("Check that your API key matches the key configured in Docker.")
    raise

except requests.exceptions.RequestException as e:
    print("Could not connect to the Retrieval API:", e)
    print("Check that Docker and the Cloudflare tunnel are running.")
    raise


# ============================================================
# 7. CONFIGURATION SUMMARY
# ============================================================

print("\n" + "=" * 55)
print("GDELT AGENTIC RAG CONFIGURATION")
print("=" * 55)

print("API URL:", API_URL)
print("GDELT API key: Configured (hidden)")
print("Gemini API key: Configured (hidden)")
print("Gemini model:", GEMINI_MODEL)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Retrieval API: Connected")
print("=" * 55)

print("\nEnvironment setup completed successfully.")

Python environment ready
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
Enter current Cloudflare API URL (https://...trycloudflare.com):   https://sparc-terrorists-possess-himself.trycloudflare.com
Enter GDELT API key (hidden): ··········
Enter Gemini API key (hidden): ··········

Checking GDELT Retrieval API connection...

GDELT Retrieval API connected successfully!
API health: {'status': 'ok', 'database': 'connected', 'total_articles': 30000}

GDELT AGENTIC RAG CONFIGURATION
API URL: https://sparc-terrorists-possess-himself.trycloudflare.com
GDELT API key: Configured (hidden)
Gemini API key: Configured (hidden)
Gemini model: gemini-3.5-flash-lite
GPU: Tesla T4
Retrieval API: Connected

Environment setup completed successfully.


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=(device == "cuda")
)
print("BGE-M3 loaded on", device)

reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=(device == "cuda")
)
print("BGE reranker loaded")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (10 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE reranker loaded


In [8]:
def gemini_text(prompt, max_retries=4):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )
            text = getattr(response, "text", None)
            if not text:
                raise RuntimeError("Gemini returned an empty response.")
            return text.strip()
        except ServerError:
            if attempt == max_retries - 1:
                raise
            wait_seconds = 3 * (attempt + 1)
            print(f"Gemini temporarily unavailable; retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

print("Gemini helper ready")

Gemini helper ready


In [9]:
def retrieve_articles(question, top_k=20):
    query_embedding = model.encode(
        [question],
        batch_size=1,
        max_length=512
    )["dense_vecs"][0].tolist()

    response = requests.post(
        f"{API_URL}/search",
        headers={"X-API-Key": API_KEY},
        json={"embedding": query_embedding, "top_k": top_k},
        timeout=90
    )
    response.raise_for_status()
    return response.json().get("results", [])

# Connectivity and vector-search smoke test
_test_articles = retrieve_articles("Recent security developments in India", top_k=3)
print("Retrieved:", len(_test_articles))
for a in _test_articles[:3]:
    print("-", a.get("title"), "|", a.get("published_at"), "|", a.get("similarity"))



pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 164.59it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 52.95it/s]


Retrieved: 3
- Top security officials to meet for two-day NIA conference; Bangladesh-based terror groups, gangster networks in focus | 2026-11-11 08:06:19 | 0.5573499925432566
- India intensifies diplomatic efforts for Captain Ajay Pant's release from UK | 2026-11-11 07:09:16 | 0.5425825756965138
- India–Poland Relations: Building New Gateways for Strategic Cooperation | 2026-11-02 13:40:56 | 0.5315793686640342


In [10]:
def build_context(articles, content_chars=3000):
    blocks = []
    for i, a in enumerate(articles, 1):
        blocks.append(
            f"""ARTICLE [{i}]
Title: {a.get('title')}
Source: {a.get('source_domain')}
Published: {a.get('published_at')}
URL: {a.get('source_url')}
Content: {(a.get('content') or '')[:content_chars]}"""
        )
    return "\n\n".join(blocks)

def generate_answer(question, articles, search_query=None, extra_instruction=None):
    if not articles:
        return (
            "I couldn't find sufficiently relevant evidence in the available "
            "articles to answer this question."
        )

    prompt = f"""You are a GDELT intelligence research assistant.

User question: {question}
Search query: {search_query or question}

Retrieved articles:
{build_context(articles)}

Rules:
- Answer only from the supplied articles. Do not invent facts.
- Respect the requested geography and time period.
- Clearly distinguish actual incidents, threat assessments, and policy developments.
- Do not describe a policy meeting or threat assessment as an incident.
- Distinguish allegations and reported claims from independently confirmed facts.
- Cite each factual paragraph using article references such as [1] and [2].
- If the evidence is insufficient, say so explicitly.

Additional instruction:
{extra_instruction or "None"}

Write a concise, structured answer."""
    return gemini_text(prompt)

def baseline_rag(question, top_k=5):
    started = time.time()
    articles = retrieve_articles(question, top_k=top_k)
    answer = generate_answer(question, articles)
    return {
        "mode": "Baseline RAG",
        "answer": answer,
        "articles": articles,
        "latency": round(time.time() - started, 2),
        "workflow": ["Vector retrieval", "Answer generation"],
        "retry_count": 0
    }

print("Baseline RAG ready")

Baseline RAG ready


In [11]:
def rerank_articles(question, articles, top_k=5):
    if not articles:
        return []
    pairs = [
        [question, (a.get("title") or "") + "\n" + (a.get("content") or "")[:3000]]
        for a in articles
    ]
    scores = reranker.compute_score(pairs, normalize=False)
    if isinstance(scores, (int, float)):
        scores = [scores]

    ranked = []
    for article, score in zip(articles, scores):
        item = article.copy()
        item["reranker_score"] = float(score)
        ranked.append(item)

    ranked.sort(key=lambda x: x["reranker_score"], reverse=True)
    return ranked[:top_k]

def reranked_rag(question, candidate_k=20, top_k=5):
    started = time.time()
    candidates = retrieve_articles(question, top_k=candidate_k)
    articles = rerank_articles(question, candidates, top_k=top_k)
    answer = generate_answer(question, articles)
    return {
        "mode": "Reranked RAG",
        "answer": answer,
        "articles": articles,
        "latency": round(time.time() - started, 2),
        "workflow": ["Vector retrieval", "Cross-encoder reranking", "Answer generation"],
        "retry_count": 0
    }

print("Reranked RAG ready")

Reranked RAG ready


In [12]:
FRESHNESS_WINDOW_DAYS = 90

def asks_for_recent_information(question):
    terms = [
        "latest", "recent", "current", "today", "now",
        "this week", "this month", "ongoing", "new developments"
    ]
    q = question.lower()
    return any(term in q for term in terms)

def parse_article_date(value):
    if not value:
        return None
    value = str(value).strip()
    candidates = [value, value[:10]]
    for candidate in candidates:
        try:
            dt = datetime.fromisoformat(candidate.replace("Z", "+00:00"))
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt
        except Exception:
            pass
    for fmt in ("%Y%m%d", "%d-%m-%Y", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(value, fmt).replace(tzinfo=timezone.utc)
        except Exception:
            pass
    return None

def classify_evidence(question, articles):
    if not articles:
        return []
    prompt = f"""Classify each article for this question.
Question: {question}

Return ONLY a JSON array, one object per article, in the same order.
Fields:
- evidence_type: one of incident, threat_assessment, policy_discussion, crime, other
- geography_relevant: boolean
- relevant: boolean
- reason: short explanation

Be strict: if the question asks about incidents in India, an event outside India is not geographically relevant merely because it affects India indirectly. A conference or policy discussion is not an incident.

Articles:
{json.dumps([{"title": a.get("title"), "content": (a.get("content") or "")[:1800],
              "published_at": a.get("published_at"), "country": a.get("country")}
             for a in articles], ensure_ascii=False)}
"""
    raw = gemini_text(prompt)
    match = re.search(r"\[.*\]", raw, flags=re.S)
    if not match:
        raise ValueError("Could not parse evidence classification JSON: " + raw[:500])
    data = json.loads(match.group(0))
    if not isinstance(data, list):
        raise ValueError("Evidence classifier did not return a JSON list.")
    return data

def classify_evidence_with_freshness(question, articles):
    labels = classify_evidence(question, articles)
    recent_query = asks_for_recent_information(question)
    now = datetime.now(timezone.utc)
    enriched = []

    for idx, article in enumerate(articles):
        item = article.copy()
        label = labels[idx] if idx < len(labels) else {}
        item.update(label)

        dt = parse_article_date(item.get("published_at"))
        age = (now - dt).total_seconds() / 86400 if dt else None
        item["article_age_days"] = round(age, 1) if age is not None else None
        item["freshness_relevant"] = (
            (age is not None and 0 <= age <= FRESHNESS_WINDOW_DAYS)
            if recent_query else True
        )
        enriched.append(item)
    return enriched

print(f"Evidence and freshness helpers ready; freshness window = {FRESHNESS_WINDOW_DAYS} days")

Evidence and freshness helpers ready; freshness window = 90 days


In [13]:
def verify_answer_claims(question, answer, articles):
    if not answer or not articles:
        return {
            "claims": [],
            "all_claims_supported": False,
            "unsupported_claim_count": 0,
            "verification_summary": "No answer or source articles available for verification."
        }

    prompt = f"""You are a claim-evidence verifier.
Check each factual claim in the answer against the supplied articles.
Return ONLY valid JSON with this structure:
{{
  "claims": [
    {{
      "claim": "factual claim",
      "citation_ids": [1],
      "supported": true,
      "reason": "brief evidence-based explanation"
    }}
  ],
  "all_claims_supported": true,
  "unsupported_claim_count": 0,
  "verification_summary": "brief summary"
}}

A claim is supported only if the cited article explicitly supports it. Do not treat general topical similarity as support.

Question: {question}
Answer:
{answer}

Sources:
{build_context(articles, content_chars=2500)}
"""
    raw = gemini_text(prompt)
    match = re.search(r"\{.*\}", raw, flags=re.S)
    if not match:
        raise ValueError("Could not parse claim verification JSON: " + raw[:500])
    return json.loads(match.group(0))

print("Claim verification helper ready")

Claim verification helper ready


In [14]:
class AgentState(TypedDict, total=False):
    question: str
    search_query: str
    articles: List[Dict[str, Any]]
    answer: str
    workflow: List[str]
    retry_count: int
    evidence: List[Dict[str, Any]]
    verification: Dict[str, Any]
    latency: float

def analyze_node(state):
    prompt = f"""Create a concise vector-search query for this intelligence question.
Preserve named places, countries, entities and the requested time period.
Return only the query, no explanation.
Question: {state['question']}"""
    query = gemini_text(prompt)
    return {"search_query": query, "workflow": state.get("workflow", []) + ["Query analysis"]}

def retrieve_node(state):
    articles = retrieve_articles(state.get("search_query") or state["question"], top_k=20)
    return {"articles": articles, "workflow": state.get("workflow", []) + ["Vector retrieval"]}

def rerank_node(state):
    articles = rerank_articles(state["question"], state.get("articles", []), top_k=8)
    return {"articles": articles, "workflow": state.get("workflow", []) + ["Cross-encoder reranking"]}

def evidence_node(state):
    evidence = classify_evidence_with_freshness(state["question"], state.get("articles", []))
    recent = asks_for_recent_information(state["question"])
    selected = [
        a for a in evidence
        if a.get("relevant") and a.get("geography_relevant")
        and (a.get("freshness_relevant") if recent else True)
    ]
    return {
        "evidence": evidence,
        "articles": selected,
        "workflow": state.get("workflow", []) + ["Evidence, geography and freshness classification"]
    }

def route_evidence(state):
    if state.get("articles"):
        return "answer"
    if state.get("retry_count", 0) < 1:
        return "rewrite"
    return "answer"

def rewrite_node(state):
    prompt = f"""Rewrite this search query to find more directly relevant evidence.
Keep the requested geography and time period. Do not broaden outside the geography.
Original question: {state['question']}
Previous search query: {state.get('search_query')}
Return only the rewritten query."""
    return {
        "search_query": gemini_text(prompt),
        "retry_count": state.get("retry_count", 0) + 1,
        "workflow": state.get("workflow", []) + ["Query rewriting"]
    }

def answer_node(state):
    articles = state.get("articles", [])
    if not articles:
        answer = (
            "I couldn't find sufficiently relevant "
            + ("and recent " if asks_for_recent_information(state["question"]) else "")
            + "articles in the available dataset to answer this question reliably."
        )
    else:
        extra = (
            "Prioritize actual incidents. Do not present threat assessments or policy discussions as incidents."
            if "incident" in state["question"].lower() else
            "Separate incidents, threat assessments and policy developments into clearly labelled categories."
        )
        answer = generate_answer(
            state["question"], articles,
            search_query=state.get("search_query"),
            extra_instruction=extra
        )
    return {"answer": answer, "workflow": state.get("workflow", []) + ["Answer generation"]}

def verify_node(state):
    if not state.get("articles"):
        verification = {
            "claims": [],
            "all_claims_supported": False,
            "unsupported_claim_count": 0,
            "verification_summary": "No source articles were selected; answer is an evidence-based abstention."
        }
    else:
        verification = verify_answer_claims(
            state["question"], state["answer"], state["articles"]
        )
    return {
        "verification": verification,
        "workflow": state.get("workflow", []) + ["Claim-level verification"]
    }

graph = StateGraph(AgentState)
graph.add_node("analyze", analyze_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("rerank", rerank_node)
graph.add_node("evidence", evidence_node)
graph.add_node("rewrite", rewrite_node)
graph.add_node("answer", answer_node)
graph.add_node("verify", verify_node)

graph.add_edge(START, "analyze")
graph.add_edge("analyze", "retrieve")
graph.add_edge("retrieve", "rerank")
graph.add_edge("rerank", "evidence")
graph.add_conditional_edges("evidence", route_evidence, {
    "answer": "answer",
    "rewrite": "rewrite"
})
graph.add_edge("rewrite", "retrieve")
graph.add_edge("answer", "verify")
graph.add_edge("verify", END)

agent_graph = graph.compile()

def agentic_rag(question):
    started = time.time()
    state = agent_graph.invoke({
        "question": question,
        "retry_count": 0,
        "workflow": []
    })
    return {
        "mode": "Agentic RAG",
        "answer": state.get("answer", ""),
        "articles": state.get("articles", []),
        "evidence": state.get("evidence", []),
        "verification": state.get("verification", {}),
        "workflow": state.get("workflow", []),
        "retry_count": state.get("retry_count", 0),
        "latency": round(time.time() - started, 2)
    }

print("Freshness-aware LangGraph Agentic RAG ready")

Freshness-aware LangGraph Agentic RAG ready


In [15]:
question = "What are the latest security incidents in India?"
result = agentic_rag(question)

print("=== AGENTIC RAG ANSWER ===")
print(result["answer"])

print("\n=== RETRIEVED ARTICLES ===")
for i, article in enumerate(result["articles"], 1):
    print(f"\n{i}. {article.get('title')}")
    print("Date:", article.get("published_at"))
    print("Evidence type:", article.get("evidence_type"))
    print("Age (days):", article.get("article_age_days"))
    print("Similarity:", article.get("similarity"))
    print("URL:", article.get("source_url"))

print("\n=== WORKFLOW ===")
print(" → ".join(result["workflow"]))
print("Retries:", result["retry_count"])
print("Latency:", result["latency"], "seconds")
print("\n=== CLAIM VERIFICATION ===")
print(json.dumps(result["verification"], indent=2, ensure_ascii=False))



pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 1790.14it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 62.04it/s]


=== AGENTIC RAG ANSWER ===
Based on the provided articles, the following security incidents have been recently reported in India:

*   **Mobile Phone and Gold Chain Snatchings (Bhopal, Madhya Pradesh):** A sharp rise in mobile phone and gold chain snatchings occurred throughout October and November. Motorcycle-borne criminals targeted pedestrians, the elderly, and women across various crowded areas and residential colonies. Notably, an elderly woman was murdered during a robbery in Shahjahanabad, and multiple incidents of theft and robbery were documented across several neighborhoods despite police patrols [1].
*   **Shooting of President's Guard during a Dispute (Baghpat, Uttar Pradesh):** Saurabh Kumar, a security personnel stationed at Rashtrapati Bhavan who was home for Diwali, was shot in the head during an altercation with a neighbor on November 8. The dispute began over the bursting of firecrackers. According to police, the bullet first struck a wall, which reduced its impact; K

In [16]:
def run_gdelt_chat(question, mode):
    if not question or not question.strip():
        return "Please enter a question.", "No articles.", {}

    functions = {
        "Baseline RAG": baseline_rag,
        "Reranked RAG": reranked_rag,
        "Agentic RAG": agentic_rag
    }
    try:
        result = functions[mode](question.strip())
        articles = result.get("articles", [])
        answer = result.get("answer", "No answer generated.")

        citation_blocks = []
        for i, a in enumerate(articles, 1):
            citation_blocks.append(
                f"""### [{i}] {a.get('title') or 'Untitled'}
- **Source:** {a.get('source_domain') or 'Unknown'}
- **Published:** {a.get('published_at') or 'Unknown'}
- **Similarity:** {a.get('similarity', 'N/A')}
- **Reranker score:** {a.get('reranker_score', 'N/A')}
- **Evidence type:** {a.get('evidence_type', 'Not classified')}
- **URL:** {a.get('source_url') or 'N/A'}"""
            )
        citations = "\n\n---\n\n".join(citation_blocks) if citation_blocks else "No articles selected."

        details = {
            "Mode": mode,
            "Latency (seconds)": result.get("latency"),
            "Retrieved articles": len(articles),
            "Workflow": result.get("workflow", []),
            "Retries": result.get("retry_count", 0),
            "Claim verification": result.get("verification", {}),
            "Evidence classification": result.get("evidence", [])
        }
        return answer, citations, details
    except Exception as exc:
        return f"Request failed: {type(exc).__name__}: {exc}", "", {"error": str(exc)}

with gr.Blocks(theme=gr.themes.Soft(), title="GDELT Intelligence Agent") as demo:
    gr.Markdown("""
# 🌍 GDELT Intelligence Agent
### Agentic RAG for Global News & Security Intelligence
Compare baseline retrieval, cross-encoder reranking and LangGraph agentic reasoning.
""")
    mode = gr.Dropdown(
        ["Baseline RAG", "Reranked RAG", "Agentic RAG"],
        value="Agentic RAG",
        label="RAG mode"
    )
    question = gr.Textbox(
        label="Intelligence question",
        placeholder="Example: What are the latest security incidents in India?",
        lines=3
    )
    submit = gr.Button("Analyze", variant="primary")
    answer = gr.Markdown(label="Analysis")
    with gr.Accordion("Retrieved articles and citations", open=True):
        citations = gr.Markdown()
    with gr.Accordion("Workflow and verification details", open=False):
        details = gr.JSON()
    submit.click(run_gdelt_chat, [question, mode], [answer, citations, details])
    question.submit(run_gdelt_chat, [question, mode], [answer, citations, details])

demo.launch(share=True, debug=True)

/tmp/ipykernel_1625/483260328.py:41: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="GDELT Intelligence Agent") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e8086d8de680194ba3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e8086d8de680194ba3.gradio.live


In [17]:
EVAL_QUESTIONS = [
    "What are the latest security incidents in India?",
    "What security threats have been reported along India's borders?",
    "What recent developments have been reported about terrorism in Jammu and Kashmir?"
]

RAG_MODES = {
    "Baseline RAG": baseline_rag,
    "Reranked RAG": reranked_rag,
    "Agentic RAG": agentic_rag
}

evaluation_results = []

for question in EVAL_QUESTIONS:
    print("\nEvaluating:", question)
    for mode_name, rag_function in RAG_MODES.items():
        started = time.time()
        try:
            result = rag_function(question)
            articles = result.get("articles", [])
            answer = result.get("answer", "")
            verification = (
                verify_answer_claims(question, answer, articles)
                if articles and answer else {}
            )
            evaluation_results.append({
                "question": question,
                "mode": mode_name,
                "latency_seconds": round(time.time() - started, 2),
                "retrieved_articles": len(articles),
                "all_claims_supported": verification.get("all_claims_supported"),
                "unsupported_claim_count": verification.get("unsupported_claim_count"),
                "answer": answer,
                "verification": verification
            })
            print(
                f"  {mode_name}: {round(time.time()-started,2)}s; "
                f"articles={len(articles)}; "
                f"all_supported={verification.get('all_claims_supported')}"
            )
        except Exception as exc:
            evaluation_results.append({
                "question": question,
                "mode": mode_name,
                "latency_seconds": None,
                "retrieved_articles": 0,
                "all_claims_supported": None,
                "unsupported_claim_count": None,
                "answer": "",
                "verification": {"error": str(exc)}
            })
            print(f"  {mode_name} FAILED: {exc}")

evaluation_df = pd.DataFrame(evaluation_results)
summary_df = (
    evaluation_df.groupby("mode", dropna=False)
    .agg(
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_retrieved_articles=("retrieved_articles", "mean"),
        verified_runs=("all_claims_supported", lambda s: int((s == True).sum())),
        evaluated_runs=("all_claims_supported", "count")
    )
    .round(2)
    .reset_index()
)
display(summary_df)
display(evaluation_df[[
    "question", "mode", "latency_seconds", "retrieved_articles",
    "all_claims_supported", "unsupported_claim_count"
]])


Evaluating: What are the latest security incidents in India?




pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 1074.91it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 55.05it/s]


  Baseline RAG: 5.99s; articles=5; all_supported=True




pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 2475.98it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 62.89it/s]


  Reranked RAG: 6.92s; articles=5; all_supported=True




pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 2285.72it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 56.09it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 2307.10it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 55.99it/s]


... (11 outputs left out)



pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 2753.98it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 40.18it/s]


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 1940.91it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 55.97it/s]


  Agentic RAG: 20.53s; articles=1; all_supported=True


,mode,avg_latency_seconds,avg_retrieved_articles,verified_runs,evaluated_runs
0,Agentic RAG,16.92,0.67,2,2
1,Baseline RAG,5.09,3.33,2,2
2,Reranked RAG,6.12,5.00,3,3


,question,mode,latency_seconds,retrieved_articles,all_claims_supported,unsupported_claim_count
0,What are the latest security incidents in India?,Baseline RAG,5.99,5,True,0.0
1,What are the latest security incidents in India?,Reranked RAG,6.92,5,True,0.0
2,What are the latest security incidents in India?,Agentic RAG,20.13,0,None,NaN
3,What security threats have been reported along...,Baseline RAG,4.19,5,True,0.0
4,What security threats have been reported along...,Reranked RAG,5.57,5,True,0.0
5,What security threats have been reported along...,Agentic RAG,10.09,1,True,0.0
6,What recent developments have been reported ab...,Baseline RAG,NaN,0,None,NaN
7,What recent developments have been reported ab...,Reranked RAG,5.87,5,True,0.0
8,What recent developments have been reported ab...,Agentic RAG,20.53,1,True,0.0


In [18]:
%pip install -q "ragas==0.2.15" "langchain-google-genai>=2.0.0,<3.0.0" "langchain-huggingface>=0.1.0"

In [19]:
# ============================================================
# AGENTIC RAG — RAGAS EVALUATION
# ============================================================

import time
import pandas as pd
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


# ------------------------------------------------------------
# 1. Configure evaluation models
# ------------------------------------------------------------

ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model=GEMINI_MODEL,
        google_api_key=GEMINI_API_KEY,
        temperature=0
    )
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )
)


# ------------------------------------------------------------
# 2. Prepare benchmark questions
# ------------------------------------------------------------

# Replace these reference answers with manually reviewed,
# evidence-based answers before treating scores as final.

BENCHMARK = [
    {
        "question": "What are the latest security incidents in India?",
        "reference": "Add a manually verified reference answer based on trusted source articles."
    },
    {
        "question": "What security threats have been reported along India's borders?",
        "reference": "Add a manually verified reference answer based on trusted source articles."
    },
    {
        "question": "What recent developments have been reported about terrorism in Jammu and Kashmir?",
        "reference": "Add a manually verified reference answer based on trusted source articles."
    }
]

print("Benchmark questions:", len(BENCHMARK))


# ------------------------------------------------------------
# 3. Run Agentic RAG and collect evidence
# ------------------------------------------------------------

ragas_records = []
ragas_failures = []

for item in BENCHMARK:

    question = item["question"]
    reference = item["reference"]

    print("\nEvaluating:", question)

    started = time.time()

    try:
        result = agentic_rag(question)

        articles = result.get("articles", [])
        answer = result.get("answer", "")

        if not articles:
            ragas_failures.append({
                "question": question,
                "error": "Agentic RAG returned zero evidence articles."
            })
            print("Skipped: no retrieved evidence.")
            continue

        if not answer:
            ragas_failures.append({
                "question": question,
                "error": "Agentic RAG returned an empty answer."
            })
            print("Skipped: empty answer.")
            continue

        contexts = [
            (
                f"Title: {a.get('title', '')}\n"
                f"Date: {a.get('published_at', '')}\n"
                f"Content: {a.get('content', '')}"
            )
            for a in articles
        ]

        ragas_records.append({
            "user_input": question,
            "response": answer,
            "retrieved_contexts": contexts,
            "reference": reference,
            "latency_seconds": round(time.time() - started, 2),
            "retrieved_articles": len(articles)
        })

        print(
            f"Collected: {len(articles)} articles | "
            f"Latency: {time.time() - started:.2f}s"
        )

        # Avoid exhausting the Gemini free-tier request limit.
        time.sleep(5)

    except Exception as exc:
        ragas_failures.append({
            "question": question,
            "error": str(exc)
        })
        print("Failed:", str(exc))


# ------------------------------------------------------------
# 4. Run RAGAS metrics
# ------------------------------------------------------------

if not ragas_records:
    print("No successful Agentic RAG records available for RAGAS.")

else:
    ragas_dataset = Dataset.from_list([
        {
            "user_input": r["user_input"],
            "response": r["response"],
            "retrieved_contexts": r["retrieved_contexts"],
            "reference": r["reference"]
        }
        for r in ragas_records
    ])

    ragas_result = evaluate(
        dataset=ragas_dataset,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall
        ],
        llm=ragas_llm,
        embeddings=ragas_embeddings
    )

    ragas_df = ragas_result.to_pandas()

    print("\nAGENTIC RAG — RAGAS RESULTS")
    display(ragas_df)

    ragas_df.to_csv(
        "gdelt_agentic_ragas_results.csv",
        index=False
    )

    print("\nRAGAS evaluation completed.")


# ------------------------------------------------------------
# 5. Save run metadata and failures
# ------------------------------------------------------------

agentic_run_df = pd.DataFrame(ragas_records)
failure_df = pd.DataFrame(ragas_failures)

agentic_run_df.to_csv(
    "gdelt_agentic_run_details.csv",
    index=False
)

failure_df.to_csv(
    "gdelt_agentic_evaluation_failures.csv",
    index=False
)

print("\nSuccessful runs:", len(ragas_records))
print("Failed/skipped runs:", len(ragas_failures))

print("\nSaved:")
print("- gdelt_agentic_ragas_results.csv")
print("- gdelt_agentic_run_details.csv")
print("- gdelt_agentic_evaluation_failures.csv")

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'